In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model

In [2]:
df = pd.read_csv("../data/processed/market_data_processed.csv")
df.head()

,Date,SP500_Close,VIX_Close,SP500_LogReturn
0,2010-01-05,1136.520020,19.350000,0.003111
1,2010-01-06,1137.140015,19.160000,0.000545
2,2010-01-07,1141.689941,19.059999,0.003993
3,2010-01-08,1144.979980,18.129999,0.002878
4,2010-01-11,1146.979980,17.549999,0.001745


In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4058 entries, 0 to 4057
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             4058 non-null   datetime64[us]
 1   SP500_Close      4058 non-null   float64       
 2   VIX_Close        4058 non-null   float64       
 3   SP500_LogReturn  4058 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 126.9 KB


In [4]:
returns = df["SP500_LogReturn"].dropna().values
returns

array([ 0.00311083,  0.00054537,  0.00399322, ..., -0.00282612,
        0.00691576, -0.01043996], shape=(4058,))

# Train test split

In [5]:
split = int(len(returns) * 0.8)

train = returns[:split]
test = returns[split:]

In [6]:
type(test), type(train)

(numpy.ndarray, numpy.ndarray)

In [ ]:
# making train and test dataframes because iloc doesn't work with array
train_df = pd.DataFrame(train, columns=["SP500_LogReturn"])
test_df = pd.DataFrame(test, columns=["SP500_LogReturn"])

In [8]:
type(train_df), type(test_df)

(pandas.DataFrame, pandas.DataFrame)

# Rolling 1-Step Ahead GARCH Forecast

In [9]:
garch_forecasts = []
realized_variance = []

rolling_returns = train_df.copy()

for i in range(len(test_df)):

    model = arch_model(rolling_returns * 100, vol="Garch", p=1, q=1)
    res = model.fit(disp="off")

    forecast = res.forecast(horizon=1)
    var_pred = forecast.variance.iloc[-1, 0] / (100**2)

    garch_forecasts.append(var_pred)

    realized_variance.append(test_df.iloc[i]**2)

    rolling_returns = pd.concat([rolling_returns, test_df.iloc[i:i+1]])

# Historical Volatility Forecast

In [10]:
hv_forecasts = []

rolling_returns = train_df.copy()

for i in range(len(test_df)):

    hv = rolling_returns.rolling(20).std().iloc[-1]
    hv_forecasts.append(hv**2)

    rolling_returns = pd.concat([rolling_returns, test_df.iloc[i:i+1]])

In [11]:
vix_test = df["VIX_Close"].iloc[split:split+len(test)]

vix_variance = ((vix_test / 100) / np.sqrt(252))**2